# Sandbox Notebook for Learning Spark Ch 3

See the chapter on O'Reilly [here](https://learning.oreilly.com/library/view/learning-spark-2nd/9781492050032/ch03.html#end_to_end_dataframe_example). This code uses the SF fire incident dataset downloaded [here](https://data.sfgov.org/Public-Safety/Fire-Incidents/wr8u-xric/about_data).

In [1]:
import os

os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@21/libexec/openjdk.jdk/Contents/Home"

from pyspark.sql import SparkSession

spark = (SparkSession
  .builder
  .appName("SfFireIncidents")
  .getOrCreate())

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/01 06:07:36 WARN Utils: Your hostname, Pauls-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 172.17.25.14 instead (on interface en0)
26/09/01 06:07:36 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/01 06:07:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
from pyspark.sql.types import *

# Programmatic way to define a schema 
fire_schema = StructType([StructField('CallNumber', IntegerType(), True),
                StructField('UnitID', StringType(), True),
                StructField('IncidentNumber', IntegerType(), True),
                StructField('CallType', StringType(), True),                  
                StructField('CallDate', StringType(), True),      
                StructField('WatchDate', StringType(), True),
                StructField('ReceivedDtTm', StringType(), True),
                StructField('EntryDtTm', StringType(), True),
                StructField('DispatchDtTm', StringType(), True),
                StructField('ResponseDtTm', StringType(), True),
                StructField('OnSceneDtTm', StringType(), True),
                StructField('TransportDtTm', StringType(), True),
                StructField('HospitalDtTm', StringType(), True),
                StructField('CallFinalDisposition', StringType(), True),
                StructField('AvailableDtTm', StringType(), True),
                StructField('Address', StringType(), True),       
                StructField('City', StringType(), True),       
                StructField('Zipcode', IntegerType(), True),       
                StructField('Battalion', StringType(), True),                 
                StructField('StationArea', StringType(), True),       
                StructField('Box', StringType(), True),       
                StructField('OriginalPriority', StringType(), True),       
                StructField('Priority', StringType(), True),       
                StructField('FinalPriority', IntegerType(), True),       
                StructField('ALSUnit', BooleanType(), True),       
                StructField('CallTypeGroup', StringType(), True),
                StructField('NumAlarms', IntegerType(), True),
                StructField('UnitType', StringType(), True),
                StructField('UnitSequenceInCallDispatch', IntegerType(), True),
                StructField('FirePreventionDistrict', StringType(), True),
                StructField('SupervisorDistrict', StringType(), True),
                StructField('Neighborhood', StringType(), True),
                StructField('RowID', StringType(), True),
                StructField('Location', StringType(), True),
                StructField('DataAsOf', StringType(), True),
                StructField('DataLoadedAt', StringType(), True)])

# Use the DataFrameReader interface to read a CSV file
sf_fire_file = "/Users/paul/Downloads/Fire_Department_and_Emergency_Medical_Services_Dispatched_Calls_for_Service_20260831_small.csv"
fire_df = spark.read.csv(sf_fire_file, header=True, schema=fire_schema)

In [3]:
from pyspark.sql.functions import *

few_fire_df = (fire_df
  .select("IncidentNumber", "AvailableDtTm", "CallType") 
  .where(col("CallType") != "Medical Incident"))
few_fire_df.show(5, truncate=False)

+--------------+-----------------------+----------------------------------+
|IncidentNumber|AvailableDtTm          |CallType                          |
+--------------+-----------------------+----------------------------------+
|16036742      |2016 Apr 02 07:55:35 AM|Structure Fire / Smoke in Building|
|16039038      |2016 Apr 07 05:44:50 PM|Traffic Collision                 |
|16039249      |2016 Apr 08 10:38:49 AM|Outside Fire                      |
|16040304      |2016 Apr 10 07:36:18 PM|Structure Fire / Smoke in Building|
|16040530      |2016 Apr 11 11:57:31 AM|Other                             |
+--------------+-----------------------+----------------------------------+
only showing top 5 rows


26/09/01 06:07:39 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: Incident Number, Call Type, Available DtTm
 Schema: IncidentNumber, CallType, AvailableDtTm
Expected: IncidentNumber but found: Incident Number
CSV file: file:///Users/paul/Downloads/Fire_Department_and_Emergency_Medical_Services_Dispatched_Calls_for_Service_20260831_small.csv


In [4]:
call_types_df = (fire_df
  .select("CallType")
  .where(col("CallType").isNotNull())
  .agg(count_distinct("CallType").alias("DistinctCallTypes")))
call_types_df.show(10, truncate=False)

26/09/01 06:07:39 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: Call Type
 Schema: CallType
Expected: CallType but found: Call Type
CSV file: file:///Users/paul/Downloads/Fire_Department_and_Emergency_Medical_Services_Dispatched_Calls_for_Service_20260831_small.csv


+-----------------+
|DistinctCallTypes|
+-----------------+
|31               |
+-----------------+



In [5]:
(fire_df
  .select("CallDate", "WatchDate", "AvailableDtTm")
  .show(5, False))

+----------+----------+-----------------------+
|CallDate  |WatchDate |AvailableDtTm          |
+----------+----------+-----------------------+
|04/03/2016|04/03/2016|2016 Apr 04 12:47:29 AM|
|04/11/2016|04/11/2016|2016 Apr 11 03:06:20 PM|
|04/02/2016|04/01/2016|2016 Apr 02 07:55:35 AM|
|04/02/2016|04/02/2016|2016 Apr 02 01:16:51 PM|
|04/01/2016|04/01/2016|2016 Apr 01 01:47:39 PM|
+----------+----------+-----------------------+
only showing top 5 rows


26/09/01 06:07:40 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: Call Date, Watch Date, Available DtTm
 Schema: CallDate, WatchDate, AvailableDtTm
Expected: CallDate but found: Call Date
CSV file: file:///Users/paul/Downloads/Fire_Department_and_Emergency_Medical_Services_Dispatched_Calls_for_Service_20260831_small.csv


In [6]:
fire_ts_df = (fire_df
  .withColumn("IncidentDate", to_timestamp(col("CallDate"), "MM/dd/yyyy"))
  .drop("CallDate") 
  .withColumn("OnWatchDate", to_timestamp(col("WatchDate"), "MM/dd/yyyy"))
  .drop("WatchDate") 
  .withColumn("AvailableDtTS", to_timestamp(col("AvailableDtTm"), 
  "yyyy MMM dd hh:mm:ss a"))
  .drop("AvailableDtTm"))

# Select the converted columns
(fire_ts_df
  .select("IncidentDate", "OnWatchDate", "AvailableDtTS")
  .show(5, False))

+-------------------+-------------------+-------------------+
|IncidentDate       |OnWatchDate        |AvailableDtTS      |
+-------------------+-------------------+-------------------+
|2016-04-03 00:00:00|2016-04-03 00:00:00|2016-04-04 00:47:29|
|2016-04-11 00:00:00|2016-04-11 00:00:00|2016-04-11 15:06:20|
|2016-04-02 00:00:00|2016-04-01 00:00:00|2016-04-02 07:55:35|
|2016-04-02 00:00:00|2016-04-02 00:00:00|2016-04-02 13:16:51|
|2016-04-01 00:00:00|2016-04-01 00:00:00|2016-04-01 13:47:39|
+-------------------+-------------------+-------------------+
only showing top 5 rows


26/09/01 06:07:40 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: Call Date, Watch Date, Available DtTm
 Schema: CallDate, WatchDate, AvailableDtTm
Expected: CallDate but found: Call Date
CSV file: file:///Users/paul/Downloads/Fire_Department_and_Emergency_Medical_Services_Dispatched_Calls_for_Service_20260831_small.csv


In [7]:
fire_ts_df = (fire_ts_df
  .withColumn("ResponseDelayedinMins",
    round((to_timestamp(col("OnSceneDtTm"), "yyyy MMM dd hh:mm:ss a").cast("long")
           - to_timestamp(col("ReceivedDtTm"), "yyyy MMM dd hh:mm:ss a").cast("long")) / 60, 2)))

(fire_ts_df
  .select("ReceivedDtTm", "OnSceneDtTm", "ResponseDelayedinMins")
  .show(5, False))

+-----------------------+-----------------------+---------------------+
|ReceivedDtTm           |OnSceneDtTm            |ResponseDelayedinMins|
+-----------------------+-----------------------+---------------------+
|2016 Apr 03 11:15:12 PM|2016 Apr 03 11:35:10 PM|19.97                |
|2016 Apr 11 01:14:47 PM|2016 Apr 11 01:27:48 PM|13.02                |
|2016 Apr 02 07:48:03 AM|NULL                   |NULL                 |
|2016 Apr 02 01:08:02 PM|2016 Apr 02 01:15:07 PM|7.08                 |
|2016 Apr 01 01:20:24 PM|2016 Apr 01 01:26:00 PM|5.6                  |
+-----------------------+-----------------------+---------------------+
only showing top 5 rows


26/09/01 06:07:40 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: Received DtTm, On Scene DtTm
 Schema: ReceivedDtTm, OnSceneDtTm
Expected: ReceivedDtTm but found: Received DtTm
CSV file: file:///Users/paul/Downloads/Fire_Department_and_Emergency_Medical_Services_Dispatched_Calls_for_Service_20260831_small.csv


In [8]:
(fire_ts_df
  .select(year('IncidentDate'))
  .distinct()
  .orderBy(year('IncidentDate'))
  .show())

26/09/01 06:07:40 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: Call Date
 Schema: CallDate
Expected: CallDate but found: Call Date
CSV file: file:///Users/paul/Downloads/Fire_Department_and_Emergency_Medical_Services_Dispatched_Calls_for_Service_20260831_small.csv


+------------------+
|year(IncidentDate)|
+------------------+
|              2001|
|              2003|
|              2005|
|              2010|
|              2013|
|              2014|
|              2015|
|              2016|
|              2020|
|              2021|
|              2023|
|              2024|
|              2025|
+------------------+



In [9]:
(fire_ts_df
  .select("CallType")
  .where(col("CallType").isNotNull())
  .groupBy("CallType")
  .count()
  .orderBy("count", ascending=False)
  .show(n=10, truncate=False))

26/09/01 06:07:41 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: Call Type
 Schema: CallType
Expected: CallType but found: Call Type
CSV file: file:///Users/paul/Downloads/Fire_Department_and_Emergency_Medical_Services_Dispatched_Calls_for_Service_20260831_small.csv


+----------------------------------+------+
|CallType                          |count |
+----------------------------------+------+
|Medical Incident                  |293373|
|Alarms                            |52440 |
|Structure Fire / Smoke in Building|31811 |
|Traffic Collision                 |18071 |
|Outside Fire                      |7367  |
|Other                             |6585  |
|Citizen Assist / Service Call     |6323  |
|Water Rescue                      |3494  |
|Gas Leak (Natural and LP Gases)   |2799  |
|Electrical Hazard                 |2314  |
+----------------------------------+------+
only showing top 10 rows


In [ ]:
import pyspark.sql.functions as F
(fire_ts_df
  .select(F.sum("NumAlarms"), F.avg("ResponseDelayedinMins"),
    F.min("ResponseDelayedinMins"), F.max("ResponseDelayedinMins"))
  .show())

26/09/01 06:07:41 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: Received DtTm, On Scene DtTm, Number of Alarms
 Schema: ReceivedDtTm, OnSceneDtTm, NumAlarms
Expected: ReceivedDtTm but found: Received DtTm
CSV file: file:///Users/paul/Downloads/Fire_Department_and_Emergency_Medical_Services_Dispatched_Calls_for_Service_20260831_small.csv


+--------------+--------------------------+--------------------------+--------------------------+
|sum(NumAlarms)|avg(ResponseDelayedinMins)|min(ResponseDelayedinMins)|max(ResponseDelayedinMins)|
+--------------+--------------------------+--------------------------+--------------------------+
|        430939|         10.33587065608151|                   -713.28|                   2176.63|
+--------------+--------------------------+--------------------------+--------------------------+



26/09/01 06:48:09 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 135587 ms exceeds timeout 120000 ms
26/09/01 06:48:09 WARN SparkContext: Killing executors is not supported by current scheduler.
26/09/01 06:48:10 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:70)
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:44)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:34)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.stora